In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/final-fantasy-dialogue-scripts/ff6-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/dff-operaomnia-lost-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/world-of-ff-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/ff7-crisiscore-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/ff7-remake-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/kingsglaive-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/ff7-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/dff-operaomnia-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/dff-operaomnia-intersecting-wills-script.csv
/kaggle/input/final-fantasy-dialogue-scripts/ff5-vba-script.csv
/kaggle/input/llm-prompt-recovery-data/gemma100.csv
/kaggle/input/llm-prompt-recovery-data/gemma10000.csv
/kaggle/input/llm-prompt-recovery-data/gemma1000.csv
/kaggle/input/human-instructions-dataset-updated-json-files/wikiHow26.json
/kaggle/input/human-instructions-dataset-updated-json-files/wikiHow23.j

In [ ]:
import torch
import torch.nn as nn
import time
from torch.nn import functional as F
import os
import glob
import csv
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

output_path = '/kaggle/working'
model_savestates_folder = os.path.join(output_path, 'model_savestates')
os.makedirs(model_savestates_folder, exist_ok=True)
model_checkpoints = os.path.join(model_savestates_folder, 'checkpoints')
os.makedirs(model_checkpoints, exist_ok=True)
best_model_saves = os.path.join(model_savestates_folder, 'best_model_saves')
os.makedirs(best_model_saves, exist_ok=True)
best_of_best_path = os.path.join(output_path, 'best_model.pth')

#HYPERPARAMS
block_size = 256
batch_size = 128
max_iterations = 50000
learning_rate = 3e-4
eval_iters = 50
n_embd = 256 #total number of dimensions to be captured
n_head = 8 # how many heads are running in parallel
n_layer = 12 # no. of decoder blocks
dropout = 0.3 # randomly deactivates some neurons to avoid overfitting
checkpoint_interval = 4000 # shows train and val loss at every checkpoint_interval'th interval

print(device)

In [ ]:
datasets_root = '/kaggle/input'

allowed_chars = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?;:()[]{}'\"-\n")
text_lines = []
text = ""
csv_files = glob.glob(os.path.join(datasets_root, "**/*.csv"), recursive=True)
json_files = glob.glob(os.path.join(datasets_root, "**/*.json"), recursive=True)

for file in csv_files:
    try:
        with open(file, 'r', encoding="utf-8") as f:
            print(file, "loaded with utf-8.")
            reader = csv.reader(f)
            header = next(reader)

            for row in reader:
                if len(row) > 0:
                    text_lines.append(row[0])
    except UnicodeDecodeError:
        with open(file, 'r', encoding="latin1") as f:
            print(file, "loaded with latin1 due to utf-8 error.")
            reader = csv.reader(f)
            header = next(reader)
            
            for row in reader:
                if len(row) > 0:
                    text_lines.append(row[0])
text = "\n".join(text_lines) + "\n"                    
print("All text is loaded. Filtering for allowed characters...")
text = "".join(c for c in text if c in allowed_chars)
print("Text filtered.")
chars =  sorted(set(text))
vocab_size=len(chars)
print("Vocab Size: ", vocab_size)
print(len(text))

In [ ]:
#mapping from strings to integers
string_to_integer = {ch:i for i,ch in enumerate(chars) }
#mapping from integers to strings
integer_to_string = {i:ch for i, ch in enumerate(chars) }

def encode(s):
    return [string_to_integer[c] for c in s]
def decode(l):
    return ''.join([integer_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
n = int(0.8*len(data))
#80-20 split, 80% is train, testing is 20%
train_data = data[:n]
val_data = data[n:]
batch_size = int(batch_size)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device),y.to(device)
    return x,y
    
x,y= get_batch('train')

In [ ]:
def get_random_chunk(split):
    """
    Returns a random chunk of the dataset as a torch tensor.
    
    split: 'train' or 'val'
    """
    data_split = train_data if split == 'train' else val_data
    start_idx = torch.randint(0, len(data_split) - block_size, (1,)).item()
    chunk = data_split[start_idx : start_idx + block_size]
    return chunk.to(device)

In [ ]:
@torch.no_grad()
def loss_estimate():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    #updates weights and biases
    return out

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        #n_embed is dim(embedding)
        #n_head is number of heads
        head_size = n_embd // n_head
        # head_size is number of features each head will be capturing in multi head attention
        self.sa = MultiHeadAttention(n_head, head_size)
        # sa = SelfAttention
        self.ffwd = FeedForward(n_embd)
        # Feed Forward : Linear -> ReLU -> Linear
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        # these are used before self-attention and feedforward respectively.

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x+y)
        #self attention -> add a norm -> feed forward -> add a norm
        
        return x

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential (
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # computing attention scores
        wei= q @ k.transpose(-2,-1)*k.shape[-1]**-0.5 
        # (B, t, head_size) @ (B, head_size, T) -> (B, T, T) 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #(B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)

        # performing weighted aggregation of values
        v = self.value(x) #(B, T, head_size)
        out = torch.matmul(wei, v)
        # (B,T,T) @ (B, T, head_size) -> (B, T, head_size)
        return out

In [ ]:
class GPTLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) #norm of final layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            
    def forward(self, index, targets=None):
        B,T = index.shape
        token_emb = self.token_embedding_table(index)
        posn_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = token_emb + posn_emb 
        x = self.blocks(x)
        x= self.ln_f(x)
        logits = self.lm_head(x)
        if targets==None:
            loss = None
        else:
            # batches, time, channels
            B, T, C = logits.shape
            #.view helps us to make a tensor with those dimensions
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is a (B,T) array of indices in the present context
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            # retrive the predictions
            logits, loss = self.forward(index_cond)
            # focus on the last time-step
            logits = logits[:,-1,:] # (B,C)
            # apply softmax on logits to get the probabilities
            probs = F.softmax(logits, dim= -1) # (B,C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B,1)
            index = torch.cat((index, index_next), dim=1) # (B,T+1)
        return index

model = GPTLM(vocab_size)
m = model.to(device)
# torch.long is just int64
context = torch.zeros((1,1), dtype=torch.long, device=device)

print('loading model params...')
model = GPTLM(vocab_size).to(device)  # fresh instance

model_file = best_of_best_path

if os.path.exists(model_file):
    print("Loading existing model...")
    model.load_state_dict(torch.load(model_file, map_location=device))
    model.train()
    print("Loaded successfully!")
else:
    print("No existing model found. Saving initial model.")
    torch.save(model.state_dict(), model_file)

model.train()
print('loaded successfully.')
# gen_char = decode(m.generate(context, max_new_tokens=500)[0].tolist())
# print(gen_char)

In [ ]:
def format_time(seconds):
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hrs:02d}:{mins:02d}:{secs:02d}"

In [ ]:
#creating a pytorch optimizer

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_losses_history = []
val_losses_history = []
iterations_history = []
best_val_loss = float('inf')

tic = time.time()
last_eval_time = tic
cumulative_time = 0.0


for i in range(max_iterations):
    if i%eval_iters == 0:
        current_time = time.time()
        losses = loss_estimate()
        train_losses_history.append(losses['train'])
        val_losses_history.append(losses['val'])
        iterations_history.append(i)
        
        if i == 0:
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}')
        else:
            step_duration = current_time - last_eval_time
            cumulative_time += step_duration
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}')
            print(f'time for last {eval_iters} iterations: {step_duration:.1f}s, Total time till now: {format_time(cumulative_time)}\n')
            
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            
            best_model_path = os.path.join(best_model_saves, f'best_model_step_{i}.pth')
            best_model_data = os.path.join(best_model_saves, 'best_models_data.txt')
            
            with open(best_model_data, "a", encoding="utf-8") as f:
                f.write(f"\n best_model_step_{i}.pth has validation loss = {best_val_loss:.3f}")
                
            torch.save({
                'step': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': losses['val'],
            }, best_model_path)
            print(f' New best model saved at step {i} with val loss {best_val_loss:.3f}')
            
            torch.save(model.state_dict(), best_of_best_path)
            print(f' Best-of-the-best model updated at {best_of_best_path}')
            
        if i % max_iterations//20 == 0:
            checkpoint_data = os.path.join(model_checkpoints, 'checkpoint_data.txt')
            torch.save({
                'step': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': losses['val'],
            }, os.path.join(model_checkpoints, f"checkpoint_step_{i}.pth"))
            print(f"Saved checkpoint at step {i}")
            
            with open(checkpoint_data, "a", encoding="utf-8") as f:
                f.write(f"\n checkpoint_step_{i}.pth has validation loss = {best_val_loss:.3f}")
        last_eval_time = current_time
    # sampling a batch of data
    xb, yb = get_batch('train')

    #computing loss
    logits, loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

toc = time.time()
print(loss.item())
total_time = toc - tic
print(f'total time for {max_iterations} iterations: {format_time(total_time)}')
    
print("\n Loss Plot: ")
plt.figure(figsize=(10, 6))
plt.plot(iterations_history, train_losses_history, label='Training Loss', marker='o', linestyle='--')
plt.plot(iterations_history, val_losses_history, label='Validation Loss', marker='x', linestyle='-')
plt.title('train loss and val loss over Iterations')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()
